# How to Align Multimodal Data (Beethoven)

**Figure 3 acid test** for TimeToAlign! — 16+ timelines across all 3 domains
(Physical, Logical, Graphical) in 5 `TimelineGroups` within one `AlignmentBundle`.

**Structure:**
1. **Part I**: Build 3 recording groups (Groups 1-3) — 15 DPTs
2. **Part II**: Build Score group (Group 4) + align with recordings
3. **Part III**: Build Emerson group (Group 5) + cross-group coordinate transfer

## 0. Gold Standard Reference Values

| ID | Description | Samples | Rate | Grp |
|----|-------------|---------|------|-----|
| DPT1-5 | Normal | 11,753,638 / 11,195 / 22,389 / 45,844 / 63,965 | 44.1k / 42 / 84 / 172 / 240 | 1 |
| DPT6-10 | Mechanical | 12,426,696 / 11,836 / 23,671 / 48,469 / 67,628 | same rates | 2 |
| DPT11-15 | Exaggerated | 8,197,748 / 7,808 / 15,616 / 31,975 / 44,614 | same rates | 3 |

| Recording | Notes | Matched | Unmatched EEP | Unmatched ABC |
|-----------|-------|---------|---------------|---------------|
| Normal | 4,026 | 3,740 | 16 | 10 |
| Mechanical | 4,026 | 3,741 | 15 | 9 |
| Exaggerated | 2,820 | 2,650 | 4 | 1,100 |

## 1. Setup

In [1]:
from pathlib import Path

import pandas as pd
from PIL import Image

from timetoalign import (
    ContinuousPhysicalTimeline,
    DiscreteGraphicalTimeline,
    NumberType,
    RepoVizzLoader,
    TableMap,
    TimeUnit,
)
from timetoalign.alignment import AlignmentBundle, TimelineGroup
from timetoalign.alignment.matching import (
    match_notes_by_attributes,
    prepare_abc_notes_for_matching,
    prepare_eep_notes_for_matching,
)
from timetoalign.loader.physical.eep_notes import EepNotesLoader
from timetoalign.loader.score import TSVLoader
from timetoalign.timelines.flow import (
    FlowMode,
    ScoreFlowController,
    create_unfolded_timeline,
)
from timetoalign.timelines.types import SegmentLine

_notebook_dir = Path(".").resolve()
DATA_DIR = (
    _notebook_dir.parent.parent
    / "tests"
    / "data"
    / "score"
    / "beethoven_op18-4iv_multimodal"
)

NORMAL_XML = DATA_DIR / "StringQuartetEEP_I_Normal" / "StringQuartetEEP_I_Normal.xml"
MECHANICAL_XML = (
    DATA_DIR / "StringQuartetEEP_I_Mechanical" / "StringQuartetEEP_I_Mechanical.xml"
)
EXAGGERATED_XML = (
    DATA_DIR / "StringQuartetEEP_I_Exaggerated" / "StringQuartetEEP_I_Exaggerated.xml"
)

INSTRUMENTS = ["vln1", "vln2", "vla", "cello"]

Each EEP recording directory contains 5 modalities (audio, 3 feature types,
MoCap) plus `.notes` files with annotated note events. The function below
builds a `TimelineGroup` from one such directory via the XML manifest.
A single `RepoVizzLoader.from_file(xml_path)` call catalogues all data;
individual timelines are then created by entry lookup — no hardcoded
filenames needed.

In [2]:
def build_recording_group(xml_path, group_id, group_name, dpt_base):
    """Build a TimelineGroup from one EEP recording directory via XML manifest.

    The RepoVizzLoader reads the XML once and catalogues every data file
    in the recording directory.  Individual timelines are created by
    entry lookup — audio, Essentia descriptors, bowing gesture descriptors,
    and score annotations are all described in the manifest.

    Args:
        xml_path: Path to the recording's XML manifest file.
        group_id: ID for the TimelineGroup.
        group_name: Human-readable name for the group.
        dpt_base: Starting DPT number (e.g. 1 for dpt1-dpt5).

    Returns:
        TimelineGroup with 5 DPTs, audio DPT containing notes as a child.
    """
    rv = RepoVizzLoader.from_file(xml_path)
    n = dpt_base

    audio = rv.create_timeline("mono", tl_uid=f"dpt{n}")
    tonal = rv.create_timeline(
        "tonal.ChordsStrength.mono", tl_uid=f"dpt{n + 1}"
    )
    lowlevel = rv.create_timeline(
        "lowlevel.Dissonance.mono", tl_uid=f"dpt{n + 2}"
    )
    rhythm = rv.create_timeline(
        "rhythm.BeatsLoudness.mono", tl_uid=f"dpt{n + 3}"
    )
    mocap = rv.create_timeline(
        rv.find_descriptor("bb_angle", "vln1"),
        tl_uid=f"dpt{n + 4}",
    )

    # Notes are auto-loaded from the Score section of the XML manifest
    notes_data = rv.store.notes
    if notes_data is not None:
        from timetoalign.loader.store import SingleStore

        notes_store = SingleStore(notes_data, name="notes")
        notes_tl = notes_store.create_timeline(uid=f"{group_id}_notes")
        audio.add_child(notes_tl, offset=0, use_conversion_map=True)

    return TimelineGroup(
        id=group_id,
        name=group_name,
        timelines=[audio, tonal, lowlevel, rhythm, mocap],
    )

***
# Part I: Three Recording Groups (Groups 1-3)

Each EEP recording = 5 DPTs (audio + 3 feature types + MoCap) at different
sampling rates, all sharing the same physical duration. Note events live
as a child of the audio DPT.

## 2. Group 1: Normal Recording (DPT1-DPT5)

In [3]:
normal_group = build_recording_group(
    NORMAL_XML, "normal", "Normal Recording", dpt_base=1
)
normal_group

TimelineGroup(id='normal', n_timelines=5, n_timestamps=2, locked=False)

The audio timeline now carries the note annotations as a child:

In [4]:
normal_group.get_timeline("dpt1")

DiscretePhysicalTimeline(id='dpt1', length=11753638, unit=samples, events=0, children=1, cmaps=1)

## 3. Group 2: Mechanical Recording (DPT6-DPT10)

In [5]:
mechanical_group = build_recording_group(
    MECHANICAL_XML, "mechanical", "Mechanical Recording", dpt_base=6
)
mechanical_group

TimelineGroup(id='mechanical', n_timelines=5, n_timestamps=2, locked=False)

## 4. Group 3: Exaggerated Recording (DPT11-DPT15)

Shorter recording (~186s) — stops after measure 131.

In [6]:
exaggerated_group = build_recording_group(
    EXAGGERATED_XML,
    "exaggerated",
    "Exaggerated Recording",
    dpt_base=11,
)
exaggerated_group

TimelineGroup(id='exaggerated', n_timelines=5, n_timestamps=2, locked=False)

## 5. Part I Summary

3 groups, 15 timelines. Each audio DPT carries note events as a child
timeline, making them accessible for matching in Part II.

**Next:** Part II builds the Score group and aligns each recording via note matching.

***
# Part II: Score Group + Alignment to Recordings (Group 4)

The score group brings together three representations of the same music:

- **CLT1**: ABC v2.6 score (notes, measures, harmonies) — `ContinuousLogicalTimeline`
- **DGT1**: OMR ground truth (3,190 note heads across 22 pages) — `DiscreteGraphicalTimeline`
- **OpenScore**: OpenScore String Quartet edition (4th movement) — `ContinuousLogicalTimeline`

All three go into one `TimelineGroup`. Cross-domain coordinate transfer
(pixels ↔ quarters ↔ seconds) works automatically via linear interpolation.

## 6. CLT1: ABC v2.6 Score

In [7]:
ABC_DIR = DATA_DIR / "ABC"
abc_loader = TSVLoader.from_file(
    ABC_DIR / "n04op18-4_04.notes.tsv",
    ABC_DIR / "n04op18-4_04.measures.tsv",
    ABC_DIR / "n04op18-4_04.harmonies.tsv",
)
clt1 = abc_loader.create_timeline(uid="clt1")
clt1

ContinuousLogicalTimeline(id='clt1', length=878.5, unit=quarters, events=0, children=3, cmaps=1)

## 7. DGT1: OMR Ground Truth

The OMR data contains 3,190 note head bounding boxes across 22 score pages.
Each page has 2 systems (except the last which has 1), giving 43 system
segments in reading order. Note events use `Left` (start) and `Width`
(duration) as pixel coordinates. Each system's `onset_beats` values
provide a c-map from pixels to quarters.

**Architecture:** `SegmentLine[SegmentLine[DiscreteGraphicalTimeline]]` →
22 page `SegmentLine[DiscreteGraphicalTimeline]` segments → 2 system sub-segments each.

In [8]:
OMR_CSV = DATA_DIR / "OMR_groundtruth" / "OMR_xml_by_score" / "omr_note_heads.csv"
OMR_IMAGES = DATA_DIR / "OMR_groundtruth" / "Images"
omr_df = pd.read_csv(OMR_CSV)
IMAGE_WIDTH = Image.open(next(OMR_IMAGES.glob("*.png"))).size[0]

Build the DGT1 bottom-up: system segments →
page `SegmentLine[DiscreteGraphicalTimeline]` →
top-level `SegmentLine[SegmentLine[DiscreteGraphicalTimeline]]`.
Events and c-maps must be added **before** a timeline is locked as a child.

In [9]:
noteheads = pd.DataFrame(
    {
        "start": omr_df["Nodes.Node.Left"].astype(int),
        "end": (omr_df["Nodes.Node.Left"] + omr_df["Nodes.Node.Width"]).astype(int),
        "onset_beats": omr_df["onset_beats"].astype(float),
        "pitch": omr_df["pitch"],
        "staff_id": omr_df["staff_id"].astype(int),
        "midi_pitch": omr_df["midi_pitch_code"].astype(int),
        "top": omr_df["Nodes.Node.Top"].astype(int),
        "page": omr_df["@pageIndex"],
        "spacing_run_id": omr_df["spacing_run_id"],
    }
)

dgt1 = SegmentLine(
    length=0,
    unit=TimeUnit.pixels,
    number_type=NumberType.int,
    segment_type=SegmentLine,
    inner_segment_type=DiscreteGraphicalTimeline,
)

for page_idx, page_data in noteheads.groupby("page", sort=True):
    # Systems ordered by vertical position (top first = reading order)
    sys_top = page_data.groupby("spacing_run_id")["top"].min()
    sys_order = sys_top.sort_values().index

    page = SegmentLine(
        length=0,
        unit=TimeUnit.pixels,
        number_type=NumberType.int,
        segment_type=DiscreteGraphicalTimeline,
    )

    for sys_rank, sys_id in enumerate(sys_order):
        sys_data = page_data[page_data["spacing_run_id"] == sys_id]

        system = DiscreteGraphicalTimeline(
            length=IMAGE_WIDTH,
            uid=f"p{page_idx}_s{sys_rank}",
            name=f"Page {page_idx + 1}, System {sys_rank + 1}",
        )

        events = sys_data.drop(columns=["page", "spacing_run_id"])
        system.add_events(events.assign(event_type="Notehead").to_dict("records"))

        # C-map: pixels → quarters (deduplicated for chords at the same x)
        pairs = (
            events[["start", "onset_beats"]]
            .drop_duplicates("start")
            .sort_values("start")
        )
        if len(pairs) >= 2:
            system.add_conversion_map(
                TableMap(
                    x_values=pairs["start"].tolist(),
                    y_values=pairs["onset_beats"].tolist(),
                    source_unit="pixels",
                    target_unit="quarters",
                    uid=f"p{page_idx}_s{sys_rank}_px_to_qb",
                )
            )

        page.append_segment(system)

    dgt1.append_segment(page, name=f"page_{page_idx}")

dgt1

SegmentLine[SegmentLine[DiscreteGraphicalTimeline]](id='tl:4', length=106425, unit=pixels, events=0, children=22)

## 8. OpenScore (4th Movement Only)

The OpenScore edition covers all 4 movements. We use the flow controller
to identify section breaks (movement boundaries) and extract the 4th
movement as a child timeline.

In [10]:
OPENSCORE_DIR = DATA_DIR / "OpenScoreSQ"
os_loader = TSVLoader.from_file(
    OPENSCORE_DIR / "sq8913219.notes.tsv",
    OPENSCORE_DIR / "sq8913219.measures.tsv",
)
os_full = os_loader.create_timeline(uid="openscore_full")
os_full

ContinuousLogicalTimeline(id='openscore_full', length=2447.0, unit=quarters, events=0, children=2, cmaps=1)

The `ScoreFlowController` derives section boundaries from the score's
flow control markup. Splitting at those coordinates creates one region
per movement.

In [11]:
flow = ScoreFlowController(os_loader.store.measures)
boundaries = flow.get_section_boundary_coordinates()
os_full.create_regions_from_boundaries(
    [0, *[float(b) for b in boundaries], float(os_full.length.value)], prefix="movement"
)
openscore = os_full.create_child_from_region("movement_4", uid="openscore")
openscore

ContinuousLogicalTimeline(id='openscore', length=878.5, unit=quarters, events=3382, children=0)

The four movement regions and the extracted child timeline:

In [12]:
os_full.diagram(show={"regions", "children"})

ContinuousLogicalTimeline[openscore_full] (16089 events, 3 children, 4 regions, 1 cmaps)
                      0 ________________________________ 2447 quarters
  ├─ notes            0 _______________________________  2441 (11898 events)
  ├─ measures         0 ________________________________ 2447 (809 events)
  └─ movement_4   1568.5                     ____________ 2447 (3382 events)
  ┄ movement_1       0 ▐═════════▌                      880
  ┄ movement_2     880            ▐═══▌                 1271.5
  ┄ movement_3   1271.5                 ▐══▌             1568.5
  ┄ movement_4   1568.5                     ▐══════════▌ 2447

## 9. Score Group (Group 4)

All three score representations in one `TimelineGroup`. Cross-domain
coordinate transfer (pixels ↔ quarters) works via linear interpolation.

In [13]:
score_group = TimelineGroup(
    id="score",
    name="Score (ABC + OMR + OpenScore)",
    timelines=[clt1, dgt1, openscore],
)
score_group

TimelineGroup(id='score', n_timelines=3, n_timestamps=2, locked=False)

## 10. Aligning Recordings with the Score via Note Matching

Each EEP recording's note events (seconds, pitch, staff) are matched
against the ABC unfolded score notes (quarterbeats, pitch, staff) using
greedy sequential matching. The result: `MatchClaim` objects that
connect recording coordinates to score coordinates.

In [14]:
# Load unfolded ABC notes (the target for all three recordings)
abc_unfolded_df = pd.read_csv(ABC_DIR / "n04op18-4_04_unfolded.notes.tsv", sep="\t")
abc_prepared = prepare_abc_notes_for_matching(abc_unfolded_df)
len(abc_prepared)  # note onsets after dropping tied notes

3750

Match each recording against the score. The `source_timeline_id` and
`target_timeline_id` are the audio DPT and CLT1 respectively — these
appear in the resulting `MatchClaim` anchors.

In [15]:
match_results = {}
for xml_path, dpt_id in [
    (NORMAL_XML, "dpt1"),
    (MECHANICAL_XML, "dpt6"),
    (EXAGGERATED_XML, "dpt11"),
]:
    eep = EepNotesLoader()
    eep.load(*sorted(xml_path.parent.glob("*_align_*.notes")))
    eep_prepared = prepare_eep_notes_for_matching(eep.events.to_pandas())
    match_results[dpt_id] = match_notes_by_attributes(
        eep_prepared,
        abc_prepared,
        match_columns=["pitch", "staff"],
        source_coord_column="start",
        target_coord_column="quarterbeats_playthrough",
        source_timeline_id=dpt_id,
        target_timeline_id="clt1",
    )

normal_match = match_results["dpt1"]
mechanical_match = match_results["dpt6"]
exaggerated_match = match_results["dpt11"]

In [16]:
{
    "Normal": normal_match.summary(),
    "Mechanical": mechanical_match.summary(),
    "Exaggerated": exaggerated_match.summary(),
}

{'Normal': {'matched': 3740,
  'unmatched_source': 16,
  'unmatched_target': 10,
  'match_claims': 3740},
 'Mechanical': {'matched': 3741,
  'unmatched_source': 15,
  'unmatched_target': 9,
  'match_claims': 3741},
 'Exaggerated': {'matched': 2650,
  'unmatched_source': 4,
  'unmatched_target': 1100,
  'match_claims': 2650}}

## Part II Summary

The score group unites 3 score representations across 2 domains (Logical +
Graphical). Note matching produced MatchClaims connecting each recording
group's audio timeline to CLT1:

| Recording | Matched | Unmatched EEP | Unmatched ABC |
|-----------|---------|---------------|---------------|
| Normal | 3,740 | 16 | 10 |
| Mechanical | 3,741 | 15 | 9 |
| Exaggerated | 2,650 | 4 | 1,100 |

**Next:** Part III adds the Emerson group and demonstrates cross-group
coordinate transfer using an `AlignmentBundle`.

***
# Part III: Emerson Group + Cross-Group Transfer (Group 5)

The Emerson group connects a commercial recording to a second score
edition via segment-level alignment. Unlike the EEP groups (per-note
alignment), the Emerson recording is aligned at the level of 10
structural sections (alpha through kappa), derived from the score's
repeat structure.

- **CLT2**: ABC v1.0 ("recordings edition") score — `ContinuousLogicalTimeline`
- **DPT16**: Emerson String Quartet recording (DG 1997) — `ContinuousPhysicalTimeline`

## 11. Building Group 5: Emerson Recording

### 11.1 CLT2: Recordings Edition Score

The recordings edition uses the same measure/repeat structure as CLT1 but
was encoded independently (ABC v1.0). We load it via TSVLoader and use its
flow controller to compute the traversal map.

In [17]:
REC_DIR = DATA_DIR / "recordings"
rec_loader = TSVLoader.from_file(
    REC_DIR / "Beethoven_Op018No4-04.notes.tsv",
    REC_DIR / "Beethoven_Op018No4-04.measures.tsv",
    REC_DIR / "Beethoven_Op018No4-04.harmonies.tsv",
)
clt2 = rec_loader.create_timeline(uid="clt2")
clt2

ContinuousLogicalTimeline(id='clt2', length=876.0, unit=quarters, events=0, children=3, cmaps=1)

### 11.2 Flow Control: Inspect the Score's Repeat Structure

The `ScoreFlowController` identifies atomic sections and flow control
events (repeats, voltas) from the measure data.

In [18]:
rec_controller = ScoreFlowController(rec_loader.store.measures)
rec_controller

Compute the default flow (all repeats taken) and a single-pass flow
(no repeats, last volta only) for comparison:

In [19]:
default_flow = rec_controller.compute_flow(FlowMode.DEFAULT)
default_flow

Flow(default: 226 folded -> 291 unfolded, ratio=1.29, 10 sections)

In [20]:
single_flow = rec_controller.compute_flow(FlowMode.SINGLE_PASS)
single_flow

Flow(single: 226 folded -> 224 unfolded, ratio=0.99, 3 sections)

### 11.3 Unfolded Timeline via TraversalMap 2

Unfolding CLT2 via the default flow creates a new timeline with events
reordered and duplicated according to the repeat structure. The unfolded
timeline carries a reverse `FlowMap` for tracing back to the original.

In [21]:
clt2_unfolded = create_unfolded_timeline(clt2, default_flow, rec_controller)
clt2_unfolded

ContinuousLogicalTimeline(id='tl:68', length=1330.0, unit=quarters, events=4173, children=30)

### 11.4 DPT16: Emerson Recording Alignment

The `measureMapAudio.csv` provides a 10-segment alignment between the
unfolded score (floating measures) and the Emerson recording (seconds).
Each segment is labeled with a Greek letter (alpha through kappa).

In [22]:
ema_df = pd.read_csv(
    REC_DIR / "Beethoven_Op018No4-04_EmersonStringQuartet_DG_measureMapAudio.csv",
    sep="\t",
    index_col=0,
)
ema_df

,measure_score_start,measure_score_end,measure_unfold_start,measure_unfold_end,seconds_start,seconds_end
α,0.75,8.750,0.75,8.750,0.567007,7.381043
β,0.75,16.750,8.75,24.750,7.381043,21.823855
γ,8.75,24.750,24.75,40.750,21.823855,37.495283
δ,16.75,40.999,40.75,64.999,37.495283,59.309161
ε,25.00,39.999,65.00,79.999,59.309161,72.699388
ζ,41.00,79.750,80.00,118.750,72.699388,106.863129
η,73.75,87.750,118.75,132.750,106.863129,118.964393
θ,79.75,95.999,132.75,148.999,118.964393,132.918685
ι,88.00,94.999,149.00,155.999,132.918685,138.739229
κ,96.00,218.250,156.00,278.250,138.739229,241.823152


Create DPT16 as a `ContinuousPhysicalTimeline` in seconds, with a
`TableMap` linking the unfolded measure boundaries to audio timestamps:

In [23]:
dpt16_duration = float(ema_df["seconds_end"].iloc[-1])
dpt16 = ContinuousPhysicalTimeline(length=dpt16_duration, uid="dpt16")

# TableMap: unfolded floating measures -> seconds (boundary correspondences)
unfold_coords = ema_df["measure_unfold_start"].tolist() + [
    ema_df["measure_unfold_end"].iloc[-1]
]
seconds_coords = ema_df["seconds_start"].tolist() + [ema_df["seconds_end"].iloc[-1]]
dpt16.add_conversion_map(
    TableMap(
        x_values=seconds_coords,
        y_values=unfold_coords,
        source_unit="seconds",
        target_unit="measures",
        uid="dpt16_sec_to_fm",
    )
)
dpt16

ContinuousPhysicalTimeline(id='dpt16', length=241.823151927, unit=seconds, events=0, children=0, cmaps=1)

### 11.5 Emerson Group

Both timelines go into one group. The group uses the DPT16 c-map
boundaries as alignment anchors (seconds <-> unfolded floating measures).

In [24]:
emerson_group = TimelineGroup(
    id="emerson",
    name="Emerson Recording (DG 1997)",
    timelines=[clt2, dpt16],
)
emerson_group

TimelineGroup(id='emerson', n_timelines=2, n_timestamps=2, locked=False)

## 12. The AlignmentBundle

The bundle collects all 5 groups and connects them via MatchClaims.
Within each group, coordinate transfer uses linear interpolation.
Between groups, WarpMaps (built from MatchClaims) enable cross-domain
transfer.

In [25]:
bundle = AlignmentBundle(name="Beethoven Op.18/4 — Multimodal Alignment")

bundle.add_group(score_group)
bundle.add_group(normal_group)
bundle.add_group(mechanical_group)
bundle.add_group(exaggerated_group)
bundle.add_group(emerson_group)

for dpt_id in ["dpt1", "dpt6", "dpt11"]:
    bundle.add_match_claims(match_results[dpt_id].match_claims)

bundle

AlignmentBundle(id='bundle:AlignmentBundle_1', name='Beethoven Op.18/4 — Multimodal Alignment', timelines=20, groups=5)

Match claims per connection:

In [26]:
pd.DataFrame(
    [
        {
            "recording": name,
            "source": dpt_id,
            "target": "clt1",
            "matched": match_results[dpt_id].n_matched,
            "unmatched_source": match_results[dpt_id].n_unmatched_source,
            "unmatched_target": match_results[dpt_id].n_unmatched_target,
        }
        for name, dpt_id in [
            ("Normal", "dpt1"),
            ("Mechanical", "dpt6"),
            ("Exaggerated", "dpt11"),
        ]
    ]
).set_index("recording")

,source,target,matched,unmatched_source,unmatched_target
recording,,,,,
Normal,dpt1,clt1,3740,16,10
Mechanical,dpt6,clt1,3741,15,9
Exaggerated,dpt11,clt1,2650,4,1100


## 13. Cross-Group Coordinate Transfer

The bundle's `get_timestamp_at()` method is the primary interface for
cross-domain coordinate transfer. Given a coordinate on any timeline,
it returns corresponding coordinates on all connected timelines —
regardless of domain.

### 13.1 Inspecting CLT1's Harmony Annotations

Before transferring coordinates, let's see what harmonic events live
on CLT1. The annotations child carries all harmony labels from the
ABC score:

In [27]:
annotations_df = clt1.get_child("annotations").get_events().to_pandas()
annotations_df[["start", "name"]].head(15)

,start,name
0,0,c.i
1,9,V65
2,10,i
3,11,V
4,12,i
5,13,V
6,17,i
7,21,v.iv
8,24,viio7/V
9,25,V(64)


### 13.2 USE CASE A — Transfer a Harmony Across All Groups

The `V7` at quarterbeat 79 (m. 20) is a dominant seventh — one of the
most recognizable sonorities. Where does this moment land across all
5 groups, in every domain?

In [28]:
bundle.get_timestamp_at(79.0, "clt1", format="prefix")

MatchLine: dropped 1433 stamp(s) that do not contain source timeline 'tl:4'


MatchLine: dropped 1433 stamp(s) that do not contain source timeline 'openscore'


{'score/clt1 (quarters)': 79.0,
 'score/tl:4 (pixels)': 9570,
 'score/openscore (quarters)': 79.0,
 'normal/dpt1 (samples)': 837936,
 'normal/dpt2 (samples)': 798,
 'normal/dpt3 (samples)': 1596,
 'normal/dpt4 (samples)': 3268,
 'normal/dpt5 (samples)': 4560,
 'mechanical/dpt6 (samples)': 911828,
 'mechanical/dpt7 (samples)': 868,
 'mechanical/dpt8 (samples)': 1737,
 'mechanical/dpt9 (samples)': 3556,
 'mechanical/dpt10 (samples)': 4962,
 'exaggerated/dpt11 (samples)': 848192,
 'exaggerated/dpt12 (samples)': 808,
 'exaggerated/dpt13 (samples)': 1616,
 'exaggerated/dpt14 (samples)': 3308,
 'exaggerated/dpt15 (samples)': 4616}

The nested format groups results by `TimelineGroup`, making it easy
to see the cross-domain correspondences. Note that sample-based
coordinates (DPT1–DPT15) are integers — as they must be:

In [29]:
bundle.get_timestamp_at(79.0, "clt1", format="nested")

{'score': {'clt1 (quarters)': 79.0,
  'tl:4 (pixels)': 9570,
  'openscore (quarters)': 79.0},
 'normal': {'dpt1 (samples)': 837936,
  'dpt2 (samples)': 798,
  'dpt3 (samples)': 1596,
  'dpt4 (samples)': 3268,
  'dpt5 (samples)': 4560},
 'mechanical': {'dpt6 (samples)': 911828,
  'dpt7 (samples)': 868,
  'dpt8 (samples)': 1737,
  'dpt9 (samples)': 3556,
  'dpt10 (samples)': 4962},
 'exaggerated': {'dpt11 (samples)': 848192,
  'dpt12 (samples)': 808,
  'dpt13 (samples)': 1616,
  'dpt14 (samples)': 3308,
  'dpt15 (samples)': 4616}}

### 13.3 Verification: Listening to the Transferred Timestamps

The timestamps above give sample counts.  Converting to seconds via
each audio timeline's `SamplesToSeconds` C-map yields seek positions
you can verify in any audio player:

In [30]:
v7_ts = bundle.get_timestamp_at(79.0, "clt1", format="flat")
pd.Series(
    {
        name: f"{float(bundle.timelines[uid].convert_to(v7_ts[k], 'seconds').value):.3f}s"
        for name, uid, k in (
            (n, u, next(k for k in v7_ts if k.startswith(u)))
            for n, u in [
                ("Normal", "dpt1"),
                ("Mechanical", "dpt6"),
                ("Exaggerated", "dpt11"),
            ]
        )
    },
    name="V7 at qb 79 — seek to",
)

Normal         19.001s
Mechanical     20.676s
Exaggerated    19.233s
Name: V7 at qb 79 — seek to, dtype: str

### 13.4 USE CASE B — Transfer Atomic Section Boundaries Across Groups

The score's repeat structure defines atomic sections (A through M).
The `ScoreFlowController` computes each section's **unfolded**
quarterbeat start coordinate — the position in the playthrough
order, which is what the bundle's WarpMaps expect:

In [31]:
abc_controller = ScoreFlowController(abc_loader.store.measures)
abc_flow = abc_controller.compute_flow(FlowMode.DEFAULT)
section_coords = abc_controller.get_atomic_section_coordinates(flow=abc_flow)
section_coords

{'A': Fraction(0, 1),
 'B': Fraction(64, 1),
 'C': Fraction(128, 1),
 'D': Fraction(192, 1),
 'E': Fraction(253, 1),
 'F': Fraction(317, 1),
 'G': Fraction(897, 2),
 'H': Fraction(993, 2),
 'I': Fraction(525, 1),
 'J': Fraction(557, 1),
 'K': Fraction(561, 1),
 'L': Fraction(589, 1),
 'M': Fraction(621, 1)}

Transfer every section boundary to all connected timelines:

In [32]:
boundary_df = pd.DataFrame(
    [
        bundle.get_timestamp_at(float(qb), "clt1", format="flat")
        for qb in section_coords.values()
    ],
    index=list(section_coords.keys()),
)
boundary_df.index.name = "section"
boundary_df

,clt1 (quarters),tl:4 (pixels),openscore (quarters),dpt1 (samples),dpt2 (samples),dpt3 (samples),dpt4 (samples),dpt5 (samples),dpt6 (samples),dpt7 (samples),dpt8 (samples),dpt9 (samples),dpt10 (samples),dpt11 (samples),dpt12 (samples),dpt13 (samples),dpt14 (samples),dpt15 (samples)
section,,,,,,,,,,,,,,,,,,
A,0.0,0.0,0.0,44100,42,84,172,240,44100,42,84,172,240,44100,42,84,172,240
B,64.0,7753.0,64.0,659662,628,1257,2573,3590,725796,691,1383,2831,3950,677580,645,1291,2643,3688
C,128.0,15506.0,128.0,1388046,1322,2644,5414,7554,1509834,1438,2876,5889,8217,1406333,1339,2679,5485,7654
D,192.0,23260.0,192.0,2055059,1957,3915,8016,11184,2232624,2126,4253,8708,12150,2063790,1966,3931,8050,11232
E,253.0,30649.0,253.0,2487974,2370,4739,9704,13540,2905988,2768,5535,11334,15815,2684332,2557,5113,10470,14609
F,317.0,38403.0,317.0,3368901,3209,6417,13140,18334,3625779,3453,6907,14142,19732,3334448,3176,6352,13006,18147
G,448.5,54333.0,448.5,4998366,4761,9521,19496,27202,5393668,5137,10274,21037,29353,4963792,4728,9456,19361,27014
H,496.5,60148.0,496.5,5252309,5003,10005,20486,28584,5669764,5400,10800,22114,30856,5208528,4961,9922,20316,28346
I,525.0,63601.0,525.0,5552728,5289,10577,21658,30219,5566614,5302,10604,21712,30294,5117077,4874,9748,19959,27848


Each row gives the exact coordinate of a section boundary in every
timeline and domain. The sample counts are integers; the seconds and
quarterbeats are floats — matching each timeline's native type.

## 14. Summary & Key Takeaways

> *"Any two events in the bundle can be related with each other — regardless of whether
> they live on the same timeline, in the same group, or even in the
> same domain — as long as a path of MatchClaims or ConversionMaps
> connects them."*

### Patterns Demonstrated

| Pattern | Example | Section |
|---------|---------|---------|
| `build_recording_group()` | Reusable factory for EEP recordings | 2-4 |
| `TSVLoader.from_file()` | Load ABC score with notes, measures, annotations | 6 |
| `SegmentLine` nesting | OMR pages → systems → noteheads | 7 |
| Region extraction | OpenScore 4-movement → movement 4 child | 8 |
| `match_notes_by_attributes()` | EEP ↔ ABC note matching | 10 |
| `ScoreFlowController.diagram()` | ASCII flow control visualization | 11 |
| `create_unfolded_timeline()` | Repeat expansion via `FlowMap` | 11 |
| `Flow.diagram()` | Flow inspection | 11 |
| `get_atomic_section_coordinates()` | Section boundaries in one call | 13 |
| `AlignmentBundle` | Multi-group cross-domain transfer | 12-13 |
| `get_timestamp_at()` | Universal coordinate transfer | 13 |

**5 groups, 23 timelines, 3 domains, 1 bundle.**